# NuevaMente — Data/IA · Corpus y Ground Truth v1

Notebook reproducible para preparar el **corpus de evaluación** y construir el **Ground Truth v1** de NuevaMente.

## Objetivos
- Cargar y validar los 10 documentos de referencia y las 50 preguntas de evaluación.
- Descargar/capturar las fuentes aprobadas y normalizarlas a Markdown.
- Generar el **Chunking Baseline v1** usado exclusivamente para evaluación.
- Construir y validar el **Ground Truth v1** con evidencia trazable a `chunk_id`.
- Exportar y versionar los artefactos congelados de evaluación.

## Alcance de arquitectura
Este notebook pertenece a **Data/IA** y prepara los artefactos de evaluación. El pipeline productivo de extracción, limpieza, chunking, embeddings multilingües, Vector Store y retrieval corresponde al equipo de **Agentes**. Data/IA consumirá posteriormente los resultados Top-K de Agentes para calcular métricas y evaluar calidad.

> **Importante:** esta versión conserva el Ground Truth ya validado: **50/50 casos**, sin cambiar las asignaciones finales ni las preguntas sustituidas durante la revisión manual.

## 0. Requisitos y estructura esperada

Dependencias mínimas (instalarlas desde `requirements.txt`, no desde el notebook): `pandas`, `openpyxl`, `requests`, `beautifulsoup4`.

Estructura esperada:

```text
project/
├── data/
│   ├── raw/
│   ├── processed/
│   └── evaluation/
│       └── NuevaMente_Dataset_ES_10_Documentos_Evaluacion_v1.xlsx
└── notebooks/
    └── 01_data_ai_corpus_ground_truth_v1.ipynb
```

Si el notebook se ejecuta desde `notebooks/`, la celda de configuración detecta automáticamente la raíz del proyecto.

In [ ]:
from pathlib import Path
from datetime import datetime
from html import unescape
import hashlib
import json
import re

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ------------------------------------------------------------
# Rutas del proyecto
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EVAL_DIR = DATA_DIR / "evaluation"

DATASET_PATH = EVAL_DIR / "NuevaMente_Dataset_ES_10_Documentos_Evaluacion_v1.xlsx"
CHUNKS_PATH = EVAL_DIR / "chunks_v1.csv"
GROUND_TRUTH_PATH = EVAL_DIR / "ground_truth_v1.csv"
GROUND_TRUTH_XLSX_PATH = EVAL_DIR / "ground_truth_v1.xlsx"
MANIFEST_PATH = EVAL_DIR / "evaluation_manifest_v1.json"

for path in [RAW_DIR, PROCESSED_DIR, EVAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Parámetros congelados del Chunking Baseline v1
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

# Hashes obtenidos en la versión v1 ya validada.
EXPECTED_CHUNKS_SHA256 = "309167963f1d5255374f3cb71b5533cfdcd97f02c7fd9650a0387c18903190c0"
EXPECTED_GROUND_TRUTH_SHA256 = "d42b6c8e94d2e80789c406651e937cf4759407059b2e54de966d6f5d1cc6731b"

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)

## 1. Carga y validación del dataset de evaluación

El archivo base contiene 10 documentos y 50 preguntas (5 por documento). La validación inicial detecta duplicados, faltantes y distribuciones inesperadas antes de continuar.

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset esperado: {DATASET_PATH}\n"
        "Colócalo en data/evaluation/ antes de ejecutar el notebook."
    )

docs = pd.read_excel(DATASET_PATH, sheet_name="Documentos_10")
eval_df = pd.read_excel(DATASET_PATH, sheet_name="Evaluacion_50")

assert docs["document_id"].nunique() == 10, "Deben existir 10 document_id únicos."
assert len(eval_df) == 50, "Deben existir exactamente 50 casos de evaluación."
assert eval_df["case_id"].nunique() == 50, "Los case_id deben ser únicos."
assert not docs["document_id"].duplicated().any(), "Hay document_id duplicados."
assert not eval_df["case_id"].duplicated().any(), "Hay case_id duplicados."

questions_per_document = eval_df.groupby("document_id")["case_id"].count()
assert (questions_per_document == 5).all(), "Cada documento debe tener exactamente 5 preguntas."

critical_columns = ["case_id", "document_id", "categoria", "pregunta", "respuesta_esperada"]
assert eval_df[critical_columns].notna().all().all(), "Hay nulos en columnas críticas."

print("✅ Dataset inicial validado")
print("Documentos:", len(docs))
print("Casos de evaluación:", len(eval_df))
display(docs["categoria"].value_counts().rename_axis("categoria").reset_index(name="documentos"))

## 2. Captura de fuentes aprobadas

Se guardan los HTML **como bytes** para preservar correctamente la codificación original. Si un archivo ya existe, se reutiliza para favorecer reproducibilidad. Usa `FORCE_DOWNLOAD = True` solo cuando el equipo decida actualizar formalmente el corpus y crear una nueva versión.

In [ ]:
category_folders = {
    "IA": "ai",
    "Backend": "backend",
    "Data Science": "data_science",
    "Frontend": "frontend",
    "Cloud / DevOps": "cloud_devops",
}

file_names = {
    "AI-ES-001": "AI-ES-001_microsoft_ai_fundamentals.html",
    "AI-ES-002": "AI-ES-002_microsoft_generative_ai.html",
    "BE-ES-001": "BE-ES-001_fastapi_primeros_pasos.html",
    "BE-ES-002": "BE-ES-002_django_tutorial_01.html",
    "DS-ES-001": "DS-ES-001_python_datos.html",
    "DS-ES-002": "DS-ES-002_analisis_datos.html",
    "FE-ES-001": "FE-ES-001_mdn_html.html",
    "FE-ES-002": "FE-ES-002_mdn_javascript.html",
    "CLD-ES-001": "CLD-ES-001_kubernetes.html",
    "CLD-ES-002": "CLD-ES-002_azure_cloud_concepts.html",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/153 Safari/537.36"
    )
}
REQUEST_TIMEOUT = 30
FORCE_DOWNLOAD = False


def capture_sources(docs_df, force_download=False):
    """Descarga o reutiliza las 10 fuentes aprobadas y devuelve un reporte."""
    results = []

    for _, row in docs_df.iterrows():
        document_id = row["document_id"]
        category = row["categoria"]
        url = row["url"]

        output_dir = RAW_DIR / category_folders[category]
        output_dir.mkdir(parents=True, exist_ok=True)
        output_path = output_dir / file_names[document_id]

        if output_path.exists() and not force_download:
            results.append({
                "document_id": document_id,
                "status": "OK",
                "status_code": None,
                "bytes": output_path.stat().st_size,
                "archivo": str(output_path),
                "origen": "cache_local",
            })
            continue

        try:
            response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            output_path.write_bytes(response.content)

            results.append({
                "document_id": document_id,
                "status": "OK",
                "status_code": response.status_code,
                "bytes": len(response.content),
                "archivo": str(output_path),
                "origen": "descarga",
            })
        except Exception as exc:
            results.append({
                "document_id": document_id,
                "status": "ERROR",
                "status_code": None,
                "bytes": 0,
                "archivo": str(output_path),
                "origen": "error",
                "error": str(exc),
            })

    return pd.DataFrame(results)


download_report = capture_sources(docs, force_download=FORCE_DOWNLOAD)
display(download_report)
assert len(download_report) == 10
assert (download_report["status"] == "OK").all(), "Hay fuentes que no pudieron capturarse."

## 3. Normalización HTML → Markdown

La normalización conserva títulos, párrafos, listas y bloques de código; elimina navegación, scripts, estilos y ruido editorial conocido. Esta transformación corresponde al **corpus de evaluación v1**, no al pipeline productivo de Agentes.

In [ ]:
def clean_whitespace(text):
    """Normaliza espacios y saltos de línea."""
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def find_main_content(soup):
    """Localiza la zona principal del artículo; usa body como fallback."""
    selectors = [
        "main", "article", "[role='main']", ".content", ".main-content",
        ".article", ".article-content", ".document", ".docs-content",
    ]
    for selector in selectors:
        content = soup.select_one(selector)
        if content:
            return content
    return soup.body if soup.body else soup


def remove_known_noise(text):
    """Elimina ruido editorial o de navegación sin romper Markdown."""
    # Lista preservada exactamente de la versión v1 validada.
    noise_phrases = [
        "Esta traducción fue hecha por IA guiada por humanos",
        "Podría tener errores al interpretar el significado original",
        "o sonar poco natural",
        "Puedes mejorar esta traducción",
        "ayudándonos a guiar mejor al LLM de IA",
        "Versión en inglés",
        "Copiar al portapapeles",
        "Editar esta página",
        "Ver código fuente",
    ]

    clean_lines = []
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped:
            clean_lines.append("")
            continue
        if stripped.startswith("```") or stripped.startswith("#"):
            clean_lines.append(line)
            continue
        if any(phrase.lower() in stripped.lower() for phrase in noise_phrases):
            continue
        clean_lines.append(line)

    return re.sub(r"\n{3,}", "\n\n", "\n".join(clean_lines)).strip()


def clean_code_text(element):
    """Extrae un bloque <pre> preservando saltos de línea."""
    raw_text = unescape(element.get_text("", strip=False))
    cleaned_lines = []

    for line in raw_text.splitlines():
        line_soup = BeautifulSoup(line, "html.parser")
        cleaned_lines.append(line_soup.get_text("", strip=False).rstrip())

    while cleaned_lines and not cleaned_lines[0].strip():
        cleaned_lines.pop(0)
    while cleaned_lines and not cleaned_lines[-1].strip():
        cleaned_lines.pop()

    return "\n".join(cleaned_lines)


def html_to_markdown(html_content):
    """Convierte HTML a Markdown conservando estructura útil para evaluación RAG."""
    soup = BeautifulSoup(html_content, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "aside", "noscript", "form", "button", "svg"]):
        tag.decompose()

    main_content = find_main_content(soup)
    lines = []

    for element in main_content.find_all(["h1", "h2", "h3", "h4", "p", "li", "pre", "code"]):
        if element.name == "code" and element.find_parent("pre") is not None:
            continue
        if element.name != "pre" and element.find_parent("pre") is not None:
            continue

        if element.name == "pre":
            code_text = clean_code_text(element)
            if code_text:
                lines.append(f"```text\n{code_text}\n```")
            continue

        text = clean_whitespace(element.get_text(" ", strip=True)).replace("¶", "").strip()
        if not text:
            continue

        prefix = {"h1": "# ", "h2": "## ", "h3": "### ", "h4": "#### ", "li": "- "}.get(element.name, "")
        if element.name == "code":
            lines.append(f"`{text}`")
        else:
            lines.append(f"{prefix}{text}")

    markdown = re.sub(r"\n{3,}", "\n\n", "\n\n".join(lines))
    return remove_known_noise(markdown).strip()

In [ ]:
normalization_results = []

for _, row in docs.iterrows():
    document_id = row["document_id"]
    category = row["categoria"]
    raw_path = RAW_DIR / category_folders[category] / file_names[document_id]

    if not raw_path.exists():
        raise FileNotFoundError(f"Archivo raw no encontrado: {raw_path}")

    html_bytes = raw_path.read_bytes()
    clean_text = html_to_markdown(html_bytes)
    output_path = PROCESSED_DIR / f"{document_id}.md"
    output_path.write_text(clean_text, encoding="utf-8")

    raw_text = BeautifulSoup(html_bytes, "html.parser").get_text(" ", strip=True)
    raw_chars = len(raw_text)
    clean_chars = len(clean_text)
    reduction_pct = round((1 - clean_chars / raw_chars) * 100, 2) if raw_chars else None

    normalization_results.append({
        "document_id": document_id,
        "status": "OK",
        "raw_chars": raw_chars,
        "clean_chars": clean_chars,
        "reduction_pct": reduction_pct,
        "archivo": str(output_path),
    })

normalization_report = pd.DataFrame(normalization_results)
display(normalization_report)

assert len(normalization_report) == 10, "Deben procesarse 10 documentos."
assert (normalization_report["status"] == "OK").all(), "Hay documentos con errores."
assert (normalization_report["clean_chars"] > 500).all(), "Algún documento quedó demasiado corto."
print("✅ Los 10 documentos fueron normalizados correctamente.")

## 4. Chunking Baseline v1

Parámetros congelados: `CHUNK_SIZE = 800` caracteres y `CHUNK_OVERLAP = 120` caracteres aproximados por bloques completos. Los bloques Markdown se mantienen intactos aunque un chunk puntual pueda superar 800 caracteres.

> Este chunking se conserva para reproducir el Ground Truth v1. El chunking productivo pertenece al equipo de Agentes.

In [ ]:
def split_markdown_into_chunks(text, chunk_size=800, overlap=120):
    """Divide Markdown respetando bloques separados por líneas vacías."""
    blocks = re.split(r"\n\s*\n", text.strip())
    chunks = []
    current_blocks = []
    current_size = 0

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        block_size = len(block)
        if current_blocks and current_size + block_size + 2 > chunk_size:
            chunks.append("\n\n".join(current_blocks))

            overlap_blocks = []
            overlap_size = 0
            for previous_block in reversed(current_blocks):
                if overlap_size + len(previous_block) > overlap:
                    break
                overlap_blocks.insert(0, previous_block)
                overlap_size += len(previous_block) + 2

            current_blocks = overlap_blocks
            current_size = sum(len(b) + 2 for b in current_blocks)

        current_blocks.append(block)
        current_size += block_size + 2

    if current_blocks:
        chunks.append("\n\n".join(current_blocks))

    return chunks


def build_chunks(docs_df):
    """Genera chunks deterministas en el orden del catálogo de documentos."""
    records = []

    for _, doc in docs_df.iterrows():
        document_id = doc["document_id"]
        processed_path = PROCESSED_DIR / f"{document_id}.md"
        text = processed_path.read_text(encoding="utf-8")
        document_chunks = split_markdown_into_chunks(text, CHUNK_SIZE, CHUNK_OVERLAP)

        for chunk_index, chunk_text in enumerate(document_chunks, start=1):
            records.append({
                "chunk_id": f"{document_id}_CH_{chunk_index:03d}",
                "document_id": document_id,
                "categoria": doc["categoria"],
                "titulo_documento": doc["titulo"],
                "chunk_index": chunk_index,
                "char_count": len(chunk_text),
                "chunk_text": chunk_text,
            })

    return pd.DataFrame(records)


chunks_df = build_chunks(docs)
print("Chunks generados antes de deduplicar:", len(chunks_df))

In [ ]:
# Control de codificación y duplicados
bad_patterns = ["Ã", "Â", "â€", "ðŸ"]
bad_chunks = chunks_df[
    chunks_df["chunk_text"].apply(lambda text: any(pattern in str(text) for pattern in bad_patterns))
]

chunks_df = (
    chunks_df
    .drop_duplicates(subset=["document_id", "chunk_text"], keep="first")
    .copy()
)

# Regenerar índices e IDs después de deduplicar, exactamente como en la v1 validada.
final_records = []
for document_id, group in chunks_df.groupby("document_id", sort=False):
    group = group.sort_values("chunk_index").copy()
    for new_index, (_, row) in enumerate(group.iterrows(), start=1):
        row = row.copy()
        row["chunk_index"] = new_index
        row["chunk_id"] = f"{document_id}_CH_{new_index:03d}"
        final_records.append(row)

chunks_df = pd.DataFrame(final_records).reset_index(drop=True)
duplicates_final = chunks_df[
    chunks_df.duplicated(subset=["document_id", "chunk_text"], keep=False)
]

assert chunks_df["document_id"].nunique() == 10
assert chunks_df["chunk_id"].is_unique
assert len(duplicates_final) == 0
assert len(bad_chunks) == 0

print("✅ Chunking Baseline v1 validado")
print("Total chunks:", len(chunks_df))
print("Documentos:", chunks_df["document_id"].nunique())
print("Tamaño medio:", round(chunks_df["char_count"].mean(), 2))

In [ ]:
chunks_df.to_csv(CHUNKS_PATH, index=False, encoding="utf-8-sig")
print("✅ Exportado:", CHUNKS_PATH)

## 5. Construcción del Ground Truth v1

### Criterio de anotación
Para cada pregunta se conserva el **conjunto mínimo de chunks suficiente** para justificar la respuesta esperada. Un chunk candidato no se convierte automáticamente en Ground Truth: la asignación final fue revisada manualmente.

Durante la revisión se sustituyeron preguntas cuya respuesta original no estaba explícitamente respaldada por el corpus procesado. Esta decisión se conserva en la columna `notas` para trazabilidad.

In [ ]:
ground_truth_df = eval_df.copy()

# Evitar errores de dtype al escribir texto en columnas cargadas como float/NaN.
for col in ["retrieved_chunk_ids", "notas"]:
    if col not in ground_truth_df.columns:
        ground_truth_df[col] = ""
    ground_truth_df[col] = ground_truth_df[col].astype("object")

ground_truth_df["relevant_chunk_ids"] = ""
ground_truth_df["num_relevant_chunks"] = 0
ground_truth_df["ground_truth_status"] = "Pendiente"

for col in ["relevant_chunk_ids", "ground_truth_status", "notas", "retrieved_chunk_ids"]:
    ground_truth_df[col] = ground_truth_df[col].fillna("").astype("object")


def assign_relevant_chunks(df, assignments):
    """Aplica asignaciones manuales de Ground Truth sin alterar otras columnas."""
    for case_id, relevant_ids in assignments.items():
        mask = df["case_id"] == case_id
        if mask.sum() != 1:
            raise ValueError(f"case_id no único o inexistente: {case_id}")
        df.loc[mask, "relevant_chunk_ids"] = ";".join(relevant_ids)
        df.loc[mask, "num_relevant_chunks"] = len(relevant_ids)
        df.loc[mask, "ground_truth_status"] = "Validado"
    return df


def replace_case(df, case_id, *, pregunta, respuesta_esperada, seccion_evidencia,
                 palabras_clave_evidencia, relevant_chunk_ids, nota):
    """Sustituye una pregunta no respaldada y conserva la razón en `notas`."""
    mask = df["case_id"] == case_id
    if mask.sum() != 1:
        raise ValueError(f"case_id no único o inexistente: {case_id}")

    df.loc[mask, "pregunta"] = pregunta
    df.loc[mask, "respuesta_esperada"] = respuesta_esperada
    df.loc[mask, "seccion_evidencia"] = seccion_evidencia
    df.loc[mask, "palabras_clave_evidencia"] = palabras_clave_evidencia
    df.loc[mask, "relevant_chunk_ids"] = ";".join(relevant_chunk_ids)
    df.loc[mask, "num_relevant_chunks"] = len(relevant_chunk_ids)
    df.loc[mask, "ground_truth_status"] = "Validado"
    df.loc[mask, "notas"] = nota
    return df

In [ ]:
GROUND_TRUTH_ASSIGNMENTS = {
    # IA — AI-ES-001
    "AI-ES-001-Q01": ["AI-ES-001_CH_001"],
    "AI-ES-001-Q02": ["AI-ES-001_CH_001"],
    "AI-ES-001-Q03": ["AI-ES-001_CH_002"],
    "AI-ES-001-Q04": ["AI-ES-001_CH_002"],

    # IA — AI-ES-002
    "AI-ES-002-Q01": ["AI-ES-002_CH_001"],
    "AI-ES-002-Q02": ["AI-ES-002_CH_001"],
    "AI-ES-002-Q03": ["AI-ES-002_CH_001"],
    "AI-ES-002-Q04": ["AI-ES-002_CH_002"],
    "AI-ES-002-Q05": ["AI-ES-002_CH_001", "AI-ES-002_CH_002"],

    # Backend — FastAPI
    "BE-ES-001-Q01": ["BE-ES-001_CH_010"],
    "BE-ES-001-Q02": ["BE-ES-001_CH_011"],
    "BE-ES-001-Q03": ["BE-ES-001_CH_013"],
    "BE-ES-001-Q04": ["BE-ES-001_CH_004"],
    "BE-ES-001-Q05": ["BE-ES-001_CH_006"],

    # Backend — Django
    "BE-ES-002-Q01": ["BE-ES-002_CH_001"],
    "BE-ES-002-Q02": ["BE-ES-002_CH_001"],
    "BE-ES-002-Q03": ["BE-ES-002_CH_005"],
    "BE-ES-002-Q04": ["BE-ES-002_CH_008"],
    "BE-ES-002-Q05": ["BE-ES-002_CH_014"],

    # Data Science
    "DS-ES-001-Q01": ["DS-ES-001_CH_001"],
    "DS-ES-001-Q02": ["DS-ES-001_CH_001"],
    "DS-ES-001-Q03": ["DS-ES-001_CH_001"],
    "DS-ES-001-Q04": ["DS-ES-001_CH_001"],
    "DS-ES-002-Q01": ["DS-ES-002_CH_001"],
    "DS-ES-002-Q02": ["DS-ES-002_CH_001"],
    "DS-ES-002-Q03": ["DS-ES-002_CH_001"],
    "DS-ES-002-Q04": ["DS-ES-002_CH_001"],
    "DS-ES-002-Q05": ["DS-ES-002_CH_001"],

    # Frontend
    "FE-ES-001-Q01": ["FE-ES-001_CH_002"],
    "FE-ES-001-Q02": ["FE-ES-001_CH_001"],
    "FE-ES-001-Q03": ["FE-ES-001_CH_001"],
    "FE-ES-001-Q04": ["FE-ES-001_CH_001"],
    "FE-ES-002-Q01": ["FE-ES-002_CH_001"],
    "FE-ES-002-Q02": ["FE-ES-002_CH_002"],
    "FE-ES-002-Q05": ["FE-ES-002_CH_001"],

    # Cloud / DevOps
    "CLD-ES-001-Q01": ["CLD-ES-001_CH_001"],
    "CLD-ES-001-Q02": ["CLD-ES-001_CH_001"],
    "CLD-ES-001-Q03": ["CLD-ES-001_CH_001"],
    "CLD-ES-001-Q04": ["CLD-ES-001_CH_002"],
    "CLD-ES-001-Q05": ["CLD-ES-001_CH_002"],
    "CLD-ES-002-Q01": ["CLD-ES-002_CH_001"],
    "CLD-ES-002-Q02": ["CLD-ES-002_CH_003"],
    "CLD-ES-002-Q03": ["CLD-ES-002_CH_003"],
}

ground_truth_df = assign_relevant_chunks(ground_truth_df, GROUND_TRUTH_ASSIGNMENTS)
print("Asignaciones directas aplicadas:", len(GROUND_TRUTH_ASSIGNMENTS))

### 5.1 Preguntas sustituidas durante la revisión manual

Las siguientes sustituciones forman parte del Ground Truth v1 ya aprobado. Se preservan exactamente para no cambiar los resultados de evaluación.

In [ ]:
REPLACEMENTS = [
    {
        "case_id": "AI-ES-001-Q05",
        "pregunta": "¿Qué conocimiento previo puede ser útil antes de iniciar el módulo?",
        "respuesta_esperada": "Un conocimiento conceptual del aprendizaje automático puede ser útil, aunque no es obligatorio.",
        "seccion_evidencia": "Requisitos previos",
        "palabras_clave_evidencia": "aprendizaje automático; conocimiento conceptual; útil; no obligatorio",
        "relevant_chunk_ids": ["AI-ES-001_CH_001"],
        "nota": "Pregunta reemplazada porque la pregunta original sobre nivel principiante no estaba respaldada por los chunks procesados.",
    },
    {
        "case_id": "DS-ES-001-Q05",
        "pregunta": "¿Qué aprenderá a almacenar y manipular el estudiante en este módulo?",
        "respuesta_esperada": "Datos en Python.",
        "seccion_evidencia": "Introducción",
        "palabras_clave_evidencia": "almacenar; manipular; datos; Python",
        "relevant_chunk_ids": ["DS-ES-001_CH_001"],
        "nota": "Pregunta reemplazada porque la pregunta original sobre nivel principiante no estaba respaldada por los chunks procesados.",
    },
    {
        "case_id": "FE-ES-001-Q05",
        "pregunta": "¿Qué puede hacer una etiqueta HTML con el contenido?",
        "respuesta_esperada": "Puede marcar diferentes partes del contenido para que aparezcan o actúen de cierta manera.",
        "seccion_evidencia": "¿Qué es HTML?",
        "palabras_clave_evidencia": "etiquetas; contenido; marcar; aparecer; actuar",
        "relevant_chunk_ids": ["FE-ES-001_CH_002"],
        "nota": "Pregunta reemplazada porque la pregunta original sobre conocimientos previos no estaba respaldada por el corpus procesado.",
    },
    {
        "case_id": "FE-ES-002-Q03",
        "pregunta": "¿Para qué sirven los bucles en JavaScript?",
        "respuesta_esperada": "Permiten repetir una tarea más de una vez.",
        "seccion_evidencia": "Guías",
        "palabras_clave_evidencia": "bucles; repetir; tarea",
        "relevant_chunk_ids": ["FE-ES-002_CH_003"],
        "nota": "Pregunta reemplazada porque la respuesta original sobre funciones complejas y comportamiento dinámico no estaba explícitamente respaldada por los chunks.",
    },
    {
        "case_id": "FE-ES-002-Q04",
        "pregunta": "¿Qué ejemplo de evento en una página web menciona el documento?",
        "respuesta_esperada": "El clic de un usuario en un botón que inicia una acción, como mostrar información.",
        "seccion_evidencia": "Eventos",
        "palabras_clave_evidencia": "evento; clic; botón; acción; información",
        "relevant_chunk_ids": ["FE-ES-002_CH_006"],
        "nota": "Pregunta reemplazada porque los ejemplos originales de actualizaciones, mapas y gráficos no aparecen en los chunks procesados.",
    },
    {
        "case_id": "CLD-ES-002-Q04",
        "pregunta": "¿Qué aborda el módulo sobre los servicios en la nube?",
        "respuesta_esperada": "Los distintos tipos de servicio en la nube, junto con casos de uso y ventajas de cada tipo.",
        "seccion_evidencia": "Módulos en esta ruta de aprendizaje",
        "palabras_clave_evidencia": "tipos de servicio; nube; casos de uso; ventajas",
        "relevant_chunk_ids": ["CLD-ES-002_CH_004"],
        "nota": "Pregunta reemplazada porque IaaS, PaaS y SaaS no aparecen explícitamente en los chunks procesados.",
    },
    {
        "case_id": "CLD-ES-002-Q05",
        "pregunta": "¿Qué enseña la serie Introducción a la infraestructura en la nube?",
        "respuesta_esperada": "Conceptos básicos de la nube, una visión general de servicios de Azure y ejercicios prácticos para implementar primeros servicios.",
        "seccion_evidencia": "Introducción a la infraestructura en la nube",
        "palabras_clave_evidencia": "conceptos básicos; nube; Azure; ejercicios prácticos",
        "relevant_chunk_ids": ["CLD-ES-002_CH_002"],
        "nota": "Pregunta reemplazada porque las ventajas específicas no aparecen enumeradas en los chunks procesados.",
    },
]

for replacement in REPLACEMENTS:
    ground_truth_df = replace_case(ground_truth_df, **replacement)

print("✅ Preguntas sustituidas preservadas:", len(REPLACEMENTS))

## 6. Validación final del Ground Truth

La validación comprueba que los 50 casos estén cerrados y que todos los `relevant_chunk_ids` existan realmente en `chunks_v1`.

In [ ]:
# Validar referencias a chunks existentes
valid_chunk_ids = set(chunks_df["chunk_id"])
missing_references = []

for _, row in ground_truth_df.iterrows():
    ids = [x.strip() for x in str(row["relevant_chunk_ids"]).split(";") if x.strip()]
    for chunk_id in ids:
        if chunk_id not in valid_chunk_ids:
            missing_references.append((row["case_id"], chunk_id))

required_columns = [
    "case_id", "document_id", "categoria", "pregunta", "respuesta_esperada",
    "relevant_chunk_ids", "num_relevant_chunks", "ground_truth_status",
]

assert len(ground_truth_df) == 50
assert ground_truth_df["case_id"].nunique() == 50
assert ground_truth_df["document_id"].nunique() == 10
assert (ground_truth_df["ground_truth_status"] == "Validado").all()
assert (ground_truth_df["relevant_chunk_ids"].fillna("").str.strip() != "").all()
assert all(col in ground_truth_df.columns for col in required_columns)
assert not missing_references, f"Hay referencias a chunks inexistentes: {missing_references}"

summary = (
    ground_truth_df.groupby("document_id")
    .agg(
        total_cases=("case_id", "count"),
        validated=("ground_truth_status", lambda x: (x == "Validado").sum()),
        relevant_chunks=("num_relevant_chunks", "sum"),
    )
    .reset_index()
)

display(summary)
print("Total casos:", len(ground_truth_df))
print("Casos validados:", (ground_truth_df["ground_truth_status"] == "Validado").sum())
print("Casos pendientes:", (ground_truth_df["ground_truth_status"] != "Validado").sum())

## 7. Exportación y versionado

Se exportan CSV, Excel y manifest. Los hashes esperados corresponden a la ejecución v1 ya validada. Si los archivos raw/processed son los mismos, los hashes deben coincidir exactamente.

In [ ]:
ground_truth_df.to_csv(GROUND_TRUTH_PATH, index=False, encoding="utf-8-sig")
ground_truth_df.to_excel(GROUND_TRUTH_XLSX_PATH, index=False)


def sha256_file(path):
    with Path(path).open("rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


chunks_sha256 = sha256_file(CHUNKS_PATH)
ground_truth_sha256 = sha256_file(GROUND_TRUTH_PATH)

print("chunks_v1.csv SHA-256:", chunks_sha256)
print("ground_truth_v1.csv SHA-256:", ground_truth_sha256)

assert chunks_sha256 == EXPECTED_CHUNKS_SHA256, (
    "El hash de chunks_v1.csv cambió respecto a la v1 validada. "
    "Revisa si cambió una fuente, la normalización o el chunking."
)
assert ground_truth_sha256 == EXPECTED_GROUND_TRUTH_SHA256, (
    "El hash de ground_truth_v1.csv cambió respecto a la v1 validada. "
    "No continúes hasta revisar la diferencia."
)

print("✅ Hashes idénticos a la versión v1 validada.")

In [ ]:
manifest = {
    "version": "v1",
    "created_at": datetime.now().isoformat(),
    "scope": "data_ai_evaluation_corpus",
    "documents": int(ground_truth_df["document_id"].nunique()),
    "chunks": int(len(chunks_df)),
    "evaluation_cases": int(ground_truth_df["case_id"].nunique()),
    "validated_cases": int((ground_truth_df["ground_truth_status"] == "Validado").sum()),
    "chunking": {
        "chunk_size_chars": CHUNK_SIZE,
        "overlap_chars_approx": CHUNK_OVERLAP,
    },
    "chunks_file": CHUNKS_PATH.name,
    "chunks_sha256": chunks_sha256,
    "ground_truth_file": GROUND_TRUTH_PATH.name,
    "ground_truth_sha256": ground_truth_sha256,
    "architecture_note": (
        "Este chunking corresponde al corpus de evaluación v1. "
        "El pipeline productivo de embeddings, Vector Store y retrieval pertenece a Agentes."
    ),
}

with MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("✅ Manifest guardado:", MANIFEST_PATH)

## 8. Cierre de la Fase 1

**Resultado de referencia de la versión validada:**
- 10 documentos.
- 121 chunks después de deduplicación.
- 50 casos de evaluación.
- 50 casos validados.
- 0 pendientes.

### Siguiente fase de Data/IA
`02_data_ai_retrieval_evaluation_v1.ipynb`

Ese notebook **no implementará embeddings ni Vector Store**. Consumirá el Top-K producido por Agentes y calculará `Recall@3`, `Recall@5`, `Precision@3`, `Precision@5`, análisis de errores y métricas por categoría/documento.

In [ ]:
print("=" * 64)
print("FASE 1 COMPLETADA — DATA/IA")
print("=" * 64)
print("Documentos:", chunks_df["document_id"].nunique())
print("Chunks:", len(chunks_df))
print("Casos de evaluación:", len(ground_truth_df))
print("Casos validados:", (ground_truth_df["ground_truth_status"] == "Validado").sum())
print("\nArtefactos congelados:")
print("-", CHUNKS_PATH)
print("-", GROUND_TRUTH_PATH)
print("-", GROUND_TRUTH_XLSX_PATH)
print("-", MANIFEST_PATH)
print("\nSiguiente fase: 02_data_ai_retrieval_evaluation_v1.ipynb")
print("Dependencia: resultados Top-K del equipo de Agentes.")